# M22 · Detecting earthquakes: network coincidence, template matching and a deep-learning detector

**Part 2 · week 3 · about 12 h core**

## 한국어 요약

이 모듈에서는 상주 ML 3.9 지진 전후 세 시간의 연속 기록에서 지진을 찾아냅니다. 먼저 M20에서 만든 STA/LTA 트리거를 여러 관측소에 적용하고, 여러 관측소가 함께 트리거될 때만 사건으로 인정하는 관측망 동시 트리거 함수를 정답을 알고 있는 합성 자료로 작성한 뒤 실제 기록에 적용합니다. 다음으로 알려진 지진의 파형(템플릿)과 긴 기록 사이의 정규화 교차상관을 계산하는 함수, 중앙값 절대 편차(MAD)로 검출 문턱값을 정하는 함수, 진폭비로 상대 규모를 구하는 함수를 합성 자료로 작성하고 점검합니다. 이 함수들로 여러분이 직접 읽은 ML 3.9와 ML 1.5 지진의 피킹으로 템플릿을 만들어 세 시간의 기록을 훑고, 결과를 기상청 목록과 비교하며, 시간을 뒤집은 기록에 같은 문턱값을 적용하는 귀무 검정으로 오검출이 얼마나 생기는지 직접 잽니다. 마지막으로 SeisBench의 PhaseNet을 같은 기록에 그대로 두 번, 그리고 설정 하나를 바꾸어 실행하고, M30의 주시로 피킹을 묶는 간단한 연관 함수를 작성해 세 방법의 검출 목록을 비교합니다. 이 필수 모듈의 마지막 과제에서는 템플릿 사이의 중복 검출을 합치되 근거를 보존하고, 서로 겹치지 않는 검증 시간 구간과 오경보율·신호 주입 회수율의 검증 계획을 작성해 순천 및 2026년 상주 연구에 옮길 준비를 합니다.

## What you will be able to do

- Write a network-coincidence detector for per-station triggers and test it on made-up triggers and against ObsPy's coincidence_trigger
- Write a normalized cross-correlation of a template with a long record, a detection threshold from the median absolute deviation and a relative magnitude from amplitude ratios, each tested on made-up records whose answers are known
- Scan three hours of continuous records with templates cut from your own picks, compare the detections with the KMA catalog and measure the false-detection rate with a null test
- Run a pretrained PhaseNet on the same records (an unchanged rerun, then one changed setting), group its picks into events with an associator you write, and compare the detection lists of all methods
- Merge repeated matches across a template bank without losing provenance, and write a validation protocol with independent time windows, exposure, injection recovery and limits for both research projects

## Before you start

- Modules: M20, M21, M30
- Reading: Withers et al. (1998), the comparison of STA/LTA with the other detectors; Gibbons & Ringdal (2006), the introduction and the method (why correlation detects small events); Peng & Zhao (2009), the methods (templates, thresholds, magnitudes); Schaff & Richards (2014), the abstract and the section on relative magnitudes; Ross, Trugman, Hauksson & Shearer (2019), the whole report (what template matching adds to a catalog)
- Needs: synthetic, stp
- Key terms (see the glossary): continuous record (연속 기록), earthquake detection (지진 검출), STA/LTA (단기/장기 평균비), trigger (트리거), trigger threshold (트리거 문턱값), false trigger (오트리거, 잘못된 트리거), network coincidence (관측망 동시 트리거), coincidence extension (동시 트리거 연장 시간), template matching (템플릿 매칭), template (템플릿 (기준 파형)), cross-correlation (상호상관, 교차상관), correlation coefficient (상관 계수), normalized cross-correlation (정규화 교차상관), cosine similarity (코사인 유사도), matched filter (정합 필터), moveout (무브아웃 (도달 시각 차)), network cross-correlation (관측망 합산 상관), detection threshold (검출 문턱값), median absolute deviation (MAD; 중앙값 절대 편차), standard deviation (표준 편차), false detection (오검출), false-alarm rate (오경보율), self-detection (자기 검출), null test (귀무 검정 (널 테스트)), local magnitude (ML; 국지 규모), relative magnitude (상대 규모), amplitude ratio (진폭비), L2 norm (L2 노름), phase pick (위상 피킹, 도달 시각 읽기), pick probability (피킹 확률), pick threshold (피킹 임계값), PhaseNet (PhaseNet), deterministic computation (결정론적 계산), phase association (위상 연관), back-projected origin time (역투영 발생 시각), association window (연관 시간 창), travel time (주시), direct wave (직접파), head wave (선두파), crossover distance (교차 거리), epicentral distance (진앙 거리), causal filter (인과 필터), band-pass filter (대역 통과 필터), taper (테이퍼), sampling rate (표본화율, 샘플링 레이트), mainshock (본진), aftershock (여진), magnitude of completeness (Mc; 완전성 규모), widget (위젯), prior exposure (사전 노출), template bank (템플릿 모음), cross-template de-duplication (템플릿 간 중복 검출 병합), held-out evaluation (독립 보류 자료 평가), injection and recovery (신호 주입 및 회수), selection bias (선택 편향)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M22.yaml](../../../.sgtl/criteria/M22.yaml).

## AI use

Follow [docs/ai_policy.md](../../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M22.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019_hours"              # folder of your STP download of the three hours (module M03)
EVENT_FOLDER = "20190721020418"           # its folder name in the download (the ML 3.9's origin time in UTC)
MAINSHOCK_ID = "2019009762"               # KMA id of the ML 3.9 earthquake of 21 July 2019 (first template)
SMALL_ID = "2019009764"                   # KMA id of the ML 1.5 earthquake 6.5 minutes later (second template)
M20_PICKS = "../picks/M20_manual_picks.csv"   # your manual picks of the ML 3.9 (module M20, task 5)
SMALL_PICKS = "../picks/M22_ml15_picks.csv"   # your manual picks of the ML 1.5 (this module, Part 5)
FMIN_HZ, FMAX_HZ, CORNERS = 2.0, 10.0, 4  # causal Butterworth band-pass used for every detector here
TAPER_S = 5.0                             # length (s) of the taper at each end of the three-hour records
STA_S, LTA_S = 0.5, 10.0                  # Part 1: STA and LTA windows (s), as in M20
THR_ON, THR_OFF = 4.0, 1.5                # Part 1: trigger thresholds, as in M20
MIN_STATIONS = 3                          # Part 1: stations that must trigger together for a network detection
EXTENSION_S = 5.0                         # Part 1: coincidence extension (s)
PRE_S, P_LEN_S, S_LEN_S = 0.2, 2.0, 3.0   # Part 5: template windows start PRE_S before a pick; P and S lengths (s)
K_MAD = 8.0                               # Parts 3 and 5: detection threshold = median + K_MAD x MAD
MIN_GAP_S = 2.0                           # Parts 3 and 5: two detections closer than this (s): only the higher
MATCH_S = 2.0                             # Part 5: a detection this close (s) to a KMA origin time is that event
DEDUP_WINDOW_S = 2.0                      # Part 7: maximum separation from a group's first match (s)
CALIBRATION_WINDOW_S = (60.0, 1260.0)      # Part 7: a rehearsal window, seconds after T0 (already inspected)
EVALUATION_WINDOW_S = (7200.0, 10200.0)    # Part 7: disjoint rehearsal window, seconds after T0
VALIDATION_GUARD_S = 60.0                 # Part 7: keep windows away from templates, each other and record edges
PN_WEIGHTS = ("stead", "2")               # Part 6: PhaseNet weights (name, version) installed for module M21
PN_THRESHOLD = 0.3                        # Part 6: PhaseNet's pick threshold for P and S
PN_THRESHOLD_CHANGED = 0.1                # Part 6: the one setting you change in the third PhaseNet run
VELOCITY_MODEL = "kim1983"                # Part 6: travel times for the associator (your direct_time of M30)
ASSOC_WINDOW_S = 2.0                      # Part 6: association window (s)
ASSOC_MIN_STATIONS = 3                    # Part 6: stations an associated event needs
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M22")

import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, UTCDateTime, read
from obspy.geodetics import gps2dist_azimuth

from sgtl_tutorial import synth_detect

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)
Path(SMALL_PICKS).parent.mkdir(parents=True, exist_ok=True)
# SeisBench looks for its weights in this folder; 'bin/sgtl-install seisbench' put the course's weights there.
_ = os.environ.setdefault("SEISBENCH_CACHE_ROOT", str(st.pq.sgtl_home() / "seisbench"))

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M22-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up data whose answers are known from how they
were built (`st.reroll("M22-T1")` draws new ones) or come from another library, and on files you write from your
own three hours of records; for those they never print the values they compare with.

This is required training for both research projects: complete it after M20, M21 and M30, before searching
either project's continuous records. Budget about 12 hours, spread across weeks 3 and 4 if needed. It needs four of your
functions from those modules (`sta_lta` and `trigger_windows` of M20, `picks_from_probability` of M21 and
`direct_time` of M30): you paste each one, unchanged, where a task asks for it. It also needs your manual picks of
M20 (`work/picks/M20_manual_picks.csv`), the STP download `sangju2019_hours` (module M03: three hours of records
at eight stations) and the PhaseNet weights of M21 (`bin/sgtl-install seisbench`). Everything runs on one core in
a few minutes. Fetch this longer dataset explicitly: in a server terminal run
`bin/sgtl data fetch sangju2019_hours`, inspect the plan, repeat with `--apply`, then run
`bin/sgtl data verify sangju2019_hours` before Part 0. All seven tasks are core: use
`bin/sgtl submit M22` (dry run), then add `--apply` when it is ready. The 2019 Sangju records here are teaching
data, separate from the Sangju project. Neither project's scientific conclusions are supplied.

The module climbs the same three steps as M20. You write the parts of three detectors on made-up data whose
answers you know (Parts 1-4), you run them on three real hours and compare them with an independent list, the
KMA catalog (Part 5), and you run a production detector, unchanged and then with one setting changed (Part 6).
Part 7 turns repeated template matches into auditable candidates and rehearses a validation plan; the projects
must reserve new, unseen evaluation records before tuning any detector.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions, your picks or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI
conversations in `work/ai_log/M22.md`.

## Part 0 · Three hours of records (about 20 min)

Until now every record was cut around a known earthquake. A **continuous record** (glossary `continuous_record`)
runs without a break for hours or days, and nobody has told you what is in it. **Earthquake detection** (glossary
`earthquake_detection`) means deciding from such records that an earthquake happened, and when. Networks detect
with automatic methods and then an analyst reviews the list; a catalog can only hold what was detected, which is
one reason for its **magnitude of completeness** (glossary `magnitude_of_completeness`, M04).

Your download `sangju2019_hours` holds three hours, from one hour before to two hours after the ML 3.9
**mainshock** (glossary `mainshock`) of 21 July 2019, at the eight stations within about 45 km of **epicentral
distance** (glossary `epicentral_distance`) that record at a **sampling rate** (glossary `sampling_rate`) of 100
samples per second. The hours after a mainshock usually hold **aftershocks** (glossary `aftershock`, M05), many
of them too small for a catalog. The cell reads the records, checks that every channel is one unbroken piece,
and lists what the KMA catalog holds for this window (times in UTC; KMA, reference key `kma_catalog`).

In [ ]:
cat = st.data.kma_catalog()
main = cat.loc[cat["event_id"] == MAINSHOCK_ID].iloc[0]
EV_LAT, EV_LON, EV_DEPTH_KM = float(main["latitude"]), float(main["longitude"]), float(main["depth_km"])

SAC_DIR = st.data.root() / DATASET / "stp_download" / "SAC" / EVENT_FOLDER
if not SAC_DIR.is_dir():
    raise FileNotFoundError(f"There are no records in {SAC_DIR}. Fetch the {DATASET} data with STP first (module "
                            "M03: sgtl data fetch sangju2019_hours), or set DATASET in the parameters cell.")
raw = Stream()
for band in ("HH", "EL"):
    for path in sorted((SAC_DIR / band).glob("*.sac")):
        raw += read(str(path))
raw.sort(keys=["network", "station", "channel"])
if len({tr.id for tr in raw}) != len(raw):
    raise ValueError("Some channel comes in more than one piece (a gap in the records); this module needs "
                     "unbroken records. Fetch the data again, or ask your mentor.")
T0 = min(tr.stats.starttime for tr in raw)
T1 = max(tr.stats.endtime for tr in raw)
DELTA = raw[0].stats.delta
print(f"{len(raw)} channels from {T0} to {T1} (UTC), {raw[0].stats.sampling_rate:g} samples per second")

rows = []
for tr in raw.select(component="Z"):
    epi_km, az, _ = gps2dist_azimuth(EV_LAT, EV_LON, tr.stats.sac.stla, tr.stats.sac.stlo)
    rows.append({"station": f"{tr.stats.network}.{tr.stats.station}", "band": tr.stats.channel[:2],
                 "epi_km": epi_km / 1000.0, "azimuth_deg": az, "lat": tr.stats.sac.stla, "lon": tr.stats.sac.stlo})
stations = pd.DataFrame(rows).sort_values("epi_km", ignore_index=True)
print(stations.round({"epi_km": 1, "azimuth_deg": 0}).to_string(index=False))

in_window = cat[(cat["origin_utc"] >= pd.Timestamp(T0.datetime, tz="UTC"))
                & (cat["origin_utc"] <= pd.Timestamp(T1.datetime, tz="UTC"))]
print(f"\nKMA catalog, {T0.strftime('%H:%M:%S')} to {T1.strftime('%H:%M:%S')} UTC:")
print(in_window[["event_id", "origin_utc", "latitude", "longitude", "depth_km", "ml"]].to_string(index=False))
KMA_TIMES = {str(r.event_id): UTCDateTime(r.origin_utc.isoformat()) for r in in_window.itertuples()}
KMA_ML = {str(r.event_id): float(r.ml) for r in in_window.itertuples()}

if not Path(M20_PICKS).is_file():
    raise FileNotFoundError(f"{M20_PICKS} is missing. The first template comes from your own picks of the ML 3.9: "
                            "finish task M20-T5 of module M20 first (its picker saves the file).")
m20_picks = pd.read_csv(M20_PICKS)
print(f"\nyour M20 picks: {len(m20_picks)} rows at {m20_picks['station'].nunique()} stations; "
      f"{sorted(set(m20_picks['station']) & set(stations['station']))} are in these records")

All eight stations lie within about 45 km of the epicenter. In M30 you found that in kim1983 a source at the KMA
depth of 14 km sends a **head wave** (glossary `head_wave`) along the boundary at 15 km that arrives first beyond
about 65 km for P and 50 km for S (its **crossover distance**, glossary `crossover_distance`). Closer in, the
**direct wave** (glossary `direct_wave`) arrives first, so for a source at 14 km the first P and the first S at
all eight stations are direct waves. The crossover holds for that depth only: it shrinks fast as the source
nears the boundary (in kim1983 the S crossover is 39 km for a source at 14.8 km), so for a slightly deeper
aftershock the first S at the farthest station may be the head wave. Part 6 comes back to this, because its
associator predicts direct waves only.

Every detector here works on the same prepared copy of the records: 64-bit floats, the mean removed, the ends
**tapered** (glossary `taper`) over `TAPER_S` seconds, and a **causal** (glossary `causal_filter`) Butterworth
**band-pass** (glossary `bandpass_filter`) from `FMIN_HZ` to `FMAX_HZ`, which removes the ocean swell below 1 Hz
and the noise of the instruments far above the energy of small earthquakes. The figure shows the vertical record
of the nearest station over all three hours, with the KMA events marked.

In [ ]:
def prepare(stream):
    """A prepared copy: float64, mean removed, ends tapered over TAPER_S, causal band-pass FMIN_HZ-FMAX_HZ."""
    out = stream.copy()
    for tr in out:
        tr.data = tr.data.astype(np.float64)
    out.detrend("demean")
    out.taper(max_percentage=None, max_length=TAPER_S, type="hann")
    out.filter("bandpass", freqmin=FMIN_HZ, freqmax=FMAX_HZ, corners=CORNERS, zerophase=False)
    return out


prepared = prepare(raw)
nearest = stations.loc[0, "station"]
z_near = prepared.select(network=nearest.split(".")[0], station=nearest.split(".")[1], component="Z")[0]
fig, ax = plt.subplots(figsize=(8.0, 2.8))
ax.plot(z_near.times() / 60.0, z_near.data, color="k", lw=0.3)
for number, (event_id, when) in enumerate(KMA_TIMES.items()):
    ax.axvline((when - z_near.stats.starttime) / 60.0, color="tab:red", lw=0.8, ls="--")
    ax.text((when - z_near.stats.starttime) / 60.0, 1.0, f" ML {KMA_ML[event_id]:.1f} ",
            ha="right" if number % 2 == 0 else "left", color="tab:red", transform=ax.get_xaxis_transform(),
            va="bottom", fontsize=8)
ax.set_xlabel(f"Minutes after {T0.strftime('%H:%M:%S')} UTC")
ax.set_ylabel(f"{z_near.id} (counts)")
ax.set_ylim(-60.0 * np.std(z_near.data[:30000]), 60.0 * np.std(z_near.data[:30000]))
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "three_hours_nearest.png", dpi=150)
print(f"{z_near.id}: noise {np.std(z_near.data[:30000]):.0f} counts in the first 5 minutes (standard deviation)")

The y-axis is cut at 60 times the noise, so the ML 3.9 and its coda run far off the panel. Look for smaller
bursts that stick out of the noise. Some may be earthquakes, others are local noise near this station. One
station alone cannot tell them apart; that is where the rest of the network comes in.

## Part 1 · Network coincidence (about 60 min, then task 1)

In M20 you wrote an STA/LTA ratio (glossary `sta_lta`) and a two-threshold **trigger** (glossary `trigger`,
thresholds: glossary `trigger_threshold`) for one record. On a continuous record a single station triggers on
many things: a truck, a door, wind, a glitch. Each of those is a **false trigger** (glossary `false_trigger`). An
earthquake, in contrast, reaches all nearby stations within a few seconds. **Network coincidence** (glossary
`network_coincidence`) uses that: it declares an event only when enough stations trigger at about the same time.
Withers et al. (1998) compare such detectors; ObsPy's `coincidence_trigger` is one (Beyreuther et al. 2010).

Triggers at different stations do not start together: a wave reaches a farther station later. Two triggers
therefore belong to one event when they overlap in time, or when one starts within a short **coincidence
extension** (glossary `coincidence_extension`) after the other has ended. A chain counts as well: if A overlaps
B and B overlaps C, then A, B and C are one event even when A and C never overlap.

### M22-T1 · Network coincidence · core · about 60 min

**Goal:** write `coincidence_events(triggers, min_stations, extension_s)`: the network detections in the
triggers of many stations.

**Given:** the trigger windows of every station (from your M20 functions, turned into seconds).

**Contract:**
- Input: `triggers`, a dict `{station name: list of (on_s, off_s)}`, times in seconds after one common start
  time (any station may have an empty list); `min_stations`, a whole number (1 or more); `extension_s`, the
  coincidence extension in seconds (0 or more).
- Put the triggers of all stations into one list in time order (by `on_s`). Walk through it. The first trigger
  starts an event: its start is that trigger's `on_s`, its end that trigger's `off_s`. Each next trigger joins
  the current event when its `on_s` is at most the event's end + `extension_s`; then the event's end becomes the
  later of its end and that trigger's `off_s`. The first trigger that does not join starts the next event.
- A station counts once per event, however many of its triggers joined it.
- An event is a network detection when triggers of at least `min_stations` different stations joined it.
- Output: a list with one tuple `(start_s, end_s, stations)` per network detection, in time order: `start_s`
  and `end_s` as floats, `stations` a tuple of the station names, sorted. An empty list when there is none.
- Example: `{"XX.STA1": [(10.0, 12.0)], "XX.STA2": [(11.0, 13.0), (40.0, 41.0)], "XX.STA3": [(14.0, 15.0)]}`
  with `min_stations = 3` and `extension_s = 1.5` gives `[(10.0, 15.0, ("XX.STA1", "XX.STA2", "XX.STA3"))]`:
  STA3 starts at 14.0, within 1.5 s after the end at 13.0; the trigger at 40.0 s starts an event of its own.
- Write it yourself: `obspy.signal.trigger` (its `coincidence_trigger`), the `trigger` method of a trace or a
  stream and the pyrocko package are not allowed here.

**Done when** `st.check("M22-T1", func=coincidence_events)` passes: on made-up triggers whose events are known
from how they were built (chains, stations that trigger twice, gaps exactly at the extension, noise triggers),
and on made-up records compared with ObsPy's `coincidence_trigger`.

**Stuck?** `st.hint("M22-T1", 1)`

In [ ]:
# Before you code: what does the example give with extension_s = 0.5, and what with min_stations = 2? Then the real
# records: with your M20 settings (STA 0.5 s, LTA 10 s, thresholds 4 and 1.5) on three hours of eight stations,
# about how many network detections do you expect with 3 stations, and with 6 stations, compared with the number
# of KMA events in the window? Why?
st.predict("M22-T1", """

""")

In [ ]:
# Your sta_lta and trigger_windows of module M20 (its tasks 1 and 2): paste both here, unchanged. Part 1 runs them
# on the three hours after the check; the check of this task does not need them.
# YOUR CODE HERE
raise NotImplementedError("M22-T1: write your code here")

In [ ]:
def coincidence_events(triggers, min_stations, extension_s):
    """Network detections: chains of overlapping triggers (plus extension_s) with at least min_stations stations.

    triggers     : dict {station: list of (on_s, off_s)}, seconds after one common start time
    min_stations : smallest number of different stations of a network detection
    extension_s  : a trigger joins an event when it starts at most this long after the event's end (s)
    returns      : list of (start_s, end_s, stations) in time order; stations a sorted tuple of names
    """
    # Step 1: one list of (on, off, station) of all stations, sorted by the on time.
    # Step 2: walk through it; a trigger joins the current event when on <= end + extension_s (the end may grow);
    #         otherwise it starts a new event.
    # Step 3: keep the events with at least min_stations different stations.
    # YOUR CODE HERE
    raise NotImplementedError("M22-T1: write your code here")

In [ ]:
st.check("M22-T1", func=coincidence_events)

Now the real records. The cell runs your `sta_lta` and `trigger_windows` on the prepared vertical channel of every
station, turns the trigger samples into seconds after `T0`, and prints the number of triggers per station. Then
it runs your `coincidence_events` for every number of stations from 2 to 8 and counts the network detections
that cover a KMA event: the event's origin time lies between 15 s before the detection's start and its end (an
earthquake's triggers start a few seconds after its origin time, and a noise trigger just before them can chain
into the same detection). Last, it lists the detections with 5 or more stations.

In [ ]:
station_triggers = {}
for tr in prepared.select(component="Z"):
    rate = tr.stats.sampling_rate
    pairs = trigger_windows(sta_lta(tr.data, int(STA_S * rate), int(LTA_S * rate)), THR_ON, THR_OFF)
    shift = float(tr.stats.starttime - T0)
    station_triggers[f"{tr.stats.network}.{tr.stats.station}"] = [(shift + on / rate, shift + off / rate)
                                                                   for on, off in pairs]
print("triggers per station:", {name: len(v) for name, v in sorted(station_triggers.items())})

kma_seconds = {event_id: float(when - T0) for event_id, when in KMA_TIMES.items()}


def covers(start, end, t):
    """True when origin time t (s after T0) lies between 15 s before a detection's start and its end."""
    return start - 15.0 <= t <= end


coincidence_table = []
for needed in range(2, 9):
    found = coincidence_events(station_triggers, needed, EXTENSION_S)
    with_kma = sum(any(covers(start, end, t) for t in kma_seconds.values()) for start, end, _ in found)
    coincidence_table.append({"min_stations": needed, "network_detections": len(found),
                              "covering_a_kma_event": with_kma,
                              "per_hour": round(len(found) / ((T1 - T0) / 3600), 1)})
print(pd.DataFrame(coincidence_table).to_string(index=False))
network_found = coincidence_events(station_triggers, MIN_STATIONS, EXTENSION_S)
print(f"\nwith MIN_STATIONS = {MIN_STATIONS}: {len(network_found)} detections; those with 5 or more stations "
      "(start, start in s after T0, length, stations):")
for start, end, names in network_found:
    if len(names) >= 5:
        kma = [e for e, t in kma_seconds.items() if covers(start, end, t)]
        print(f"  {T0 + start}  {start:8.1f} s  {end - start:5.1f} s  {', '.join(names)}"
              + (f"  (KMA {kma[0]})" if kma else ""))

**Explain (3-5 sentences):** How many triggers did the noisiest and the quietest station give in three hours?
How many network detections did you get with 3 and with 6 stations, and how many of them cover a KMA event? Look
at the detections with 5 or more stations: which are the KMA events, when does the ML 3.9's detection start
compared with its origin time and why, and what might the others be? What does the table say about choosing
`min_stations` for a network of eight stations?

## Part 2 · Template matching: the normalized cross-correlation (about 60 min, then task 2)

STA/LTA asks only whether something got louder. **Template matching** (glossary `template_matching`) asks a
sharper question: does this piece of record look like a known earthquake? The **template** (glossary
`template_waveform`) is a short piece of a known earthquake's record on one channel, for example the first two
seconds of its P wave on a vertical channel. Earthquakes close to each other with similar mechanisms send similar
waveforms to a station, so a small event next to the template's earthquake looks like a scaled copy of it, even
when it hides in the noise. A detector that slides a known signal along a record is a **matched filter**
(glossary `matched_filter`). Gibbons & Ringdal (2006) and Shelly, Beroza & Ide (2007) made it a tool for finding
small events; Ross et al. (2019) used it to find about ten times as many earthquakes in southern California as the
regional catalog listed.

The measure of likeness is the **correlation coefficient** (glossary `correlation_coefficient`) of the template
with every piece of the record of the same length. Remove the mean of the template and of the piece, multiply
them sample by sample, add up, and divide by the **L2 norm** (glossary `l2_norm`, the square root of the sum of
the squared samples) of each:

$$r_i = \frac{\sum_k t'_k\, x'_{i+k}}{\sqrt{\sum_k t'^2_k}\,\sqrt{\sum_k x'^2_{i+k}}},$$

where $t'$ is the template minus its mean and $x'_{i+k}$ the record piece that starts at sample $i$ minus that
piece's own mean. It lies between -1 and 1 and does not care about size: a copy 100 times smaller in quiet noise
still gives nearly 1. One value per shift $i$ is the **normalized cross-correlation** (glossary
`normalized_cross_correlation`); the sum of products alone, without the division, is the **cross-correlation**
(glossary `cross_correlation`) that M41 uses to time arrivals.

### M22-T2 · A normalized cross-correlation · core · about 60 min

**Goal:** write `normalized_cc(template, data)`: the correlation coefficient of a template with every piece of a
record of the same length.

**Given:** two 1-D sequences of numbers.

**Contract:**
- Input: `template`, `m` numbers; `data`, `n >= m` numbers (lists, or numpy arrays of any number type, 32-bit
  integers included).
- Output: a 1-D numpy array of 64-bit floats with `n - m + 1` values; value `i` is the correlation coefficient of
  the template with `data[i:i + m]`, by the formula above (each piece minus its own mean).
- Where a piece of the data is flat (all its samples equal, so its variance is 0), the value is 0; a variance
  that rounding made slightly negative counts as 0.
- Speed: Part 5 runs it on records of 1,080,000 samples, about 80 times. A loop over the pieces in Python takes
  minutes per record; sums over sliding windows from running totals (`np.cumsum`) and a sliding product from
  `np.correlate` (or `scipy.signal.correlate`) take well under a second.
- Example: `template = [1, 2, 3]` and `data = [0, 1, 2, 3, 2, 4, 6]` give `[1.0, 1.0, 0.0, 0.5, 1.0]`.
- Write the normalization yourself. Not allowed here: anything that gives a correlation coefficient or a
  **cosine similarity** (glossary `cosine_similarity`: the coefficient without removing the means) for you.
  That is ObsPy's `obspy.signal` (its `cross_correlation` and `trigger`), `numpy.corrcoef` (also of
  `numpy.ma`), `statistics.correlation`, `scipy.stats` (`pearsonr`), `scipy.spatial.distance`, pandas' `corr`,
  `corrwith` and `autocorr` (also on rolling, expanding and ewm windows), statsmodels, scikit-learn, xarray,
  polars, dask, PyTorch, JAX, TensorFlow, Keras and CuPy, the time-series packages stumpy, mass_ts, tslearn and
  librosa, and the template matchers EQcorrscan, fast_matched_filter, scikit-image, OpenCV, pyrocko and
  PocketQuake.

**Done when** `st.check("M22-T2", func=normalized_cc)` passes: on made-up records with scaled copies of the
template at known samples (where the value must be 1, or -1 for an upside-down copy), with large offsets, flat
stretches and 32-bit integer data, compared with ObsPy's `correlate_template` (with `normalize="full"`).

**Stuck?** `st.hint("M22-T2", 1)`

In [ ]:
# Before you code: in the example, why is value 2 exactly 0 and value 3 only 0.5? A record holds a copy of the
# template 10 times smaller than the noise: will its correlation coefficient be near 1, near 0.1, or near 0? And
# a copy 10 times larger than the noise? What happens to your sums if the record has an offset of a million
# counts (raw counts often do)?
st.predict("M22-T2", """

""")

In [ ]:
def normalized_cc(template, data):
    """Correlation coefficient of the template with every piece data[i:i + m] (m = len(template)).

    template : 1-D sequence of m numbers
    data     : 1-D sequence of n >= m numbers (any number type)
    returns  : numpy float64 array of n - m + 1 values in [-1, 1]; 0 where a piece of data is flat
    """
    # Step 1: float64 arrays; the template minus its mean (then sum_k t'_k x'_(i+k) = sum_k t'_k x_(i+k)).
    # Step 2: the numerator for every i: a sliding product of the template with the data.
    # Step 3: for every piece: sum x and sum x^2 over m samples from running totals; m times its variance is
    #         sum x^2 - (sum x)^2 / m.
    # Step 4: divide by sqrt(sum t'^2 * m * variance) where that is above 0; 0 elsewhere.
    # YOUR CODE HERE
    raise NotImplementedError("M22-T2: write your code here")

In [ ]:
st.check("M22-T2", func=normalized_cc)

Your function on a made-up record of 60 s at 100 samples per second (from `synth_detect`, the course's generator
of practice data for this module): smooth noise with four copies of a made-up 2-s waveform at known times, two of
them smaller than the noise, one upside down.

In [ ]:
demo_rng = np.random.default_rng(22)
demo_wave = synth_detect.wavelet(demo_rng, 200)
demo_record, demo_truth = synth_detect.record_with_copies(
    demo_rng, demo_wave, 6000, starts=[700, 2100, 3600, 4900], scales=[5.0, 0.5, -2.0, 0.25], noise=1.0, smooth=3)
demo_cc = normalized_cc(demo_wave, demo_record)
for start, scale in zip(demo_truth["starts"], demo_truth["scales"]):
    print(f"copy at {start / 100:5.2f} s, {abs(scale) * np.abs(demo_wave).max() / np.std(demo_record):5.2f} "
          f"times the record's standard deviation{' (upside down)' if scale < 0 else ''}: "
          f"coefficient {demo_cc[start]:+.3f}")
away = np.ones(demo_cc.size, dtype=bool)
for start in demo_truth["starts"]:
    away[max(0, start - 200):start + 200] = False
print(f"largest value more than 2 s from every copy: {demo_cc[away].max():.3f}; "
      f"smallest: {demo_cc[away].min():.3f}")
fig, (ax_rec, ax_cc) = plt.subplots(2, 1, figsize=(7.5, 4.2), sharex=True)
ax_rec.plot(np.arange(demo_record.size) / 100.0, demo_record, color="k", lw=0.4)
ax_rec.set_ylabel("Made-up record")
ax_cc.plot(np.arange(demo_cc.size) / 100.0, demo_cc, color="k", lw=0.6)
ax_cc.set_ylabel("Correlation coefficient")
ax_cc.set_xlabel("Time of the piece's first sample (s); dashed: the copies")
for start in demo_truth["starts"]:
    for ax in (ax_rec, ax_cc):
        ax.axvline(start / 100.0, color="tab:red", lw=0.7, ls="--")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "normalized_cc_made_up.png", dpi=150)

**Explain (3-5 sentences):** Which copies stand out from the noise in the coefficient, and which do not? Compare
the coefficient of each copy with its size relative to the noise, and with your prediction. Why can one channel
alone not tell a small copy from a chance likeness in the noise, and what do you expect from averaging the
coefficients of many channels? What would go wrong in your function if you did not remove each piece's own mean?

## Part 3 · How high is high enough? A threshold from the MAD (about 45 min, then task 3)

One channel's coefficient wanders up and down with the noise. An earthquake, though, lines up on every channel at
once, if you allow for the **moveout** (glossary `moveout`): the wave reaches a farther station later, and S
arrives after P. Template matching therefore shifts each channel's coefficients back by that channel's lag (where
its template window starts after the template earthquake's origin time) and averages all channels: the **network
cross-correlation** (glossary `network_cross_correlation`). At the origin time of a copy of the template
earthquake all channels peak together; noise peaks do not line up, so they average away.

A **detection threshold** (glossary `detection_threshold`) decides which peaks count. It must sit above the noise,
and the noise level differs from one record and template to the next, so it is set from the network
cross-correlation itself, in units of its scatter. The **standard deviation** (glossary `standard_deviation`) is
a poor unit here: the strong earthquakes you want to find inflate it. The **median absolute deviation** (glossary
`median_absolute_deviation`, MAD, from M21) is not pulled up by a few large values:

$$\mathrm{MAD} = \mathrm{median}\,|c_i - \mathrm{median}(c)|, \qquad \text{threshold} = \mathrm{median}(c) +
k\cdot\mathrm{MAD}.$$

Every peak above it that is not an earthquake is a **false detection** (glossary `false_detection`). How many a
detector makes per hour on records that hold no event is its **false-alarm rate** (glossary `false_alarm_rate`).
The larger `k`, the fewer false detections and the fewer small earthquakes. This module starts at `K_MAD = 8`.
Two detections of one earthquake (one on each side of a peak's shoulder) are avoided with a minimum gap, exactly
as your `picks_from_probability` of M21 does for a pick-probability curve.

### M22-T3 · A threshold from the MAD · core · about 45 min

**Goal:** write `mad_detections(cc, delta, k, min_gap_s)`: the detections in a network cross-correlation above
`median + k * MAD`.

**Given:** your `picks_from_probability` of M21 (task M21-T1), which finds the peaks above a threshold with a
minimum gap.

**Contract:**
- Input: `cc`, a 1-D numpy array (one value per sample); `delta`, the sample spacing (s); `k`, a number (> 0);
  `min_gap_s`, in s.
- `MAD = median(|cc - median(cc)|)`, over all samples; `threshold = median(cc) + k * MAD`.
- The detections are the picks of your `picks_from_probability(cc, delta, threshold, min_gap_s)`: peaks (higher
  than both neighbours) at or above the threshold, never two closer than `min_gap_s` (the higher one stays).
- Output: a tuple `(times_s, values, threshold)`: the detection times in seconds after the first sample and the
  values of `cc` there (two 1-D float arrays, sorted by time), and the threshold (a float). `cc` must not change.
- Example: `cc = [0.0, 0.1, -0.1, 0.0, 0.9, 0.0, 0.05, -0.05, 0.6, 0.0]`, `delta = 0.5`, `k = 4`, `min_gap_s =
  1.0` give the threshold 0.2 (median 0, MAD 0.05) and detections at 2.0 s (0.9) and 4.0 s (0.6).
- Write the median absolute deviation yourself with numpy (`np.median`): `scipy.stats.median_abs_deviation`,
  `statsmodels`, `astropy.stats` and the peak finders of `scipy.signal` (`find_peaks` and its relatives) are not
  allowed, here or in your `picks_from_probability`.

**Done when** `st.check("M22-T3", func=mad_detections)` passes: on made-up network cross-correlations (bumps of
known height and place on smooth noise, offsets, a few huge values, pure noise from many seeds), compared with
SciPy's `median_abs_deviation` and `find_peaks`.

**Stuck?** `st.hint("M22-T3", 1)`

In [ ]:
# Before you code: for Gaussian noise the MAD is about 0.67 standard deviations. At k = 8, how many standard
# deviations is the threshold above the median? On a network cross-correlation of pure noise lasting one hour,
# about how many false detections do you expect at k = 4, 6 and 8 (with a 2-s minimum gap)? Would a threshold of
# median + 8 standard deviations find more or fewer small earthquakes in the real records than 8 MAD, and why?
st.predict("M22-T3", """

""")

In [ ]:
# Your picks_from_probability of module M21 (its task 1): paste it here, unchanged.
# YOUR CODE HERE
raise NotImplementedError("M22-T3: write your code here")

In [ ]:
def mad_detections(cc, delta, k, min_gap_s):
    """Detections in a network cross-correlation: peaks at or above median + k * MAD, at least min_gap_s apart.

    cc        : 1-D numpy array, one value per sample (it is not changed)
    delta     : sample spacing (s)
    k         : the threshold in MADs above the median
    min_gap_s : two detections closer than this (s) cannot both stay; the higher one stays
    returns   : (times_s, values, threshold): times after the first sample (s), cc there, and the threshold
    """
    # Step 1: the median of cc, then the median of the distances |cc - median|: the MAD.
    # Step 2: threshold = median + k * MAD.
    # Step 3: the peaks of your picks_from_probability with that threshold and min_gap_s.
    # YOUR CODE HERE
    raise NotImplementedError("M22-T3: write your code here")

In [ ]:
st.check("M22-T3", func=mad_detections)

Now measure the false-alarm rate. The cell makes a made-up network of 20 channels with one hour of smooth noise
each and no earthquake at all (`synth_detect.noise_channels`), correlates each channel with its own made-up
template using your `normalized_cc`, shifts each by its lag, averages them, and counts your detections for several
values of `k`. It also prints how far the largest value lies above the median, in MADs and in standard deviations.

In [ ]:
noise_rng = np.random.default_rng(2207)
noise_templates, noise_records, noise_truth = synth_detect.noise_channels(noise_rng, 20, 360_000, 200, smooth=4)
stacked = np.zeros(360_000)
for template, record, lag in zip(noise_templates, noise_records, noise_truth["lags"]):
    values = normalized_cc(template, record)
    stacked[:values.size - lag] += values[lag:]          # origin sample q takes the value of piece q + lag
stacked /= len(noise_templates)
spread = np.median(np.abs(stacked - np.median(stacked)))
print(f"pure noise, 1 hour, 20 channels: MAD {spread:.4f}, standard deviation {np.std(stacked):.4f} "
      f"(ratio {spread / np.std(stacked):.3f}); largest value {(stacked.max() - np.median(stacked)) / spread:.1f} "
      "MADs above the median")
for k_try in (3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 10.0):
    times_k, _, level_k = mad_detections(stacked, 0.01, k_try, MIN_GAP_S)
    print(f"k = {k_try:4.1f}: threshold {level_k:.4f}, {len(times_k):4d} false detections in one hour")

**Explain (3-5 sentences):** On pure noise, how many false detections per hour did you get at k = 4, 6 and 8, and
how does that compare with your prediction? How many MADs above the median did the largest noise value reach, and
what does that say about a safe `k` for one hour of this network? Why does the threshold come from the median and
the MAD of the network cross-correlation itself, rather than from a fixed value such as 0.3? What does a pure-noise
test like this one not tell you about real records?

## Part 4 · Relative magnitudes (about 30 min, then task 4)

A detection says when, not how big. The **local magnitude** (glossary `local_magnitude`) of the KMA catalog
comes from the largest amplitudes, and one magnitude unit means 10 times the amplitude. A detected event that
looks like the template can therefore borrow the template's magnitude: if it is 10 times smaller on the same
channels, it is one unit smaller. This is a **relative magnitude** (glossary `relative_magnitude`), from the
**amplitude ratio** (glossary `amplitude_ratio`) of the event and the template on each channel.

Which amplitude? Peng & Zhao (2009) took the median, over all channels, of the ratios of the largest amplitudes.
Schaff & Richards (2014) show that cross-correlation gives the least-squares amplitude ratio (the coefficient
times the ratio of the L2 norms), which is biased low when the two waveforms are not quite alike, and that the
ratio of the L2 norms alone removes that bias. This module uses their ratio of L2 norms, on each channel, and the
median over the channels, which a single odd channel cannot pull away.

### M22-T4 · A relative magnitude · core · about 30 min

**Goal:** write `relative_magnitude(template_windows, event_windows, template_ml)`: the magnitude of a detected
event from its amplitude ratios against the template.

**Given:** the template's window and the event's window on each channel (the same length on one channel), and
the template's magnitude.

**Contract:**
- Input: `template_windows` and `event_windows`, two lists of the same length (one entry per channel) of 1-D
  sequences of numbers; `template_ml`, a number.
- On each channel: remove each window's own mean; the channel's step is `log10(norm of the event window / norm of
  the template window)`, where the norm is the L2 norm (square root of the sum of squares).
- Leave out a channel where either window is flat (norm 0).
- Output: `template_ml + median of the steps` (a float; with an even number of channels the median is the mean
  of the two middle steps); `nan` when no channel is left.
- Example: `template_windows = [[1, -1, 1, -1], [2, 0, -2, 0]]`, `event_windows = [[10, -10, 10, -10], [0, 2,
  0, -2]]` and `template_ml = 1.0` give 1.5: the steps are 1.0 and 0.0.
- Not allowed: the course package (`sgtl_tutorial`).

**Done when** `st.check("M22-T4", func=relative_magnitude)` passes: on made-up windows whose sizes were set by
construction (channels scaled by known factors, offsets, flat channels, event waveforms whose shape differs from
the template's while their size is known).

**Stuck?** `st.hint("M22-T4", 1)`

In [ ]:
# Before you code: in the example, what would the least-squares ratio (the coefficient times the ratio of the L2
# norms) give for the second channel, and what magnitude would follow? An event identical to the template but 3
# times larger on every channel: what is its magnitude relative to the template? And what would you get if you
# used the ratio of the energies (sums of squares) instead of the L2 norms?
st.predict("M22-T4", """

""")

In [ ]:
def relative_magnitude(template_windows, event_windows, template_ml):
    """Magnitude of an event from the median over channels of log10(L2 norm ratio) against the template.

    template_windows : list of 1-D sequences, the template's window on each channel
    event_windows    : list of 1-D sequences, the event's window on the same channels (same lengths)
    template_ml      : magnitude of the template's earthquake
    returns          : template_ml + median of the channel steps (float); nan when no channel is left
    """
    # Step 1: on each channel, both windows minus their means; their L2 norms.
    # Step 2: skip a channel with a norm of 0; else the step is log10(event norm / template norm).
    # Step 3: template_ml plus the median of the steps.
    # YOUR CODE HERE
    raise NotImplementedError("M22-T4: write your code here")

In [ ]:
st.check("M22-T4", func=relative_magnitude)

Your function against the least-squares ratio on made-up channels: five channels of an event 10 times larger than
the template, first with the template's own waveform, then with a waveform that is only partly like it (the same
size, part template and part other wiggles), as for an event a few kilometres away or with a slightly different
mechanism. The least-squares magnitude is computed here, in the open, only for the comparison.

In [ ]:
mag_rng = np.random.default_rng(5)
shapes = [synth_detect.wavelet(mag_rng, 300) for _ in range(5)]
same = [10.0 * s for s in shapes]
other = []
for s in shapes:
    w = synth_detect.wavelet(mag_rng, 300)
    mixed = 0.6 * s / np.linalg.norm(s) + 0.8 * w / np.linalg.norm(w)      # partly like the template
    other.append(10.0 * mixed * np.linalg.norm(s) / np.linalg.norm(mixed))
for label, events in (("same waveform", same), ("other waveform", other)):
    coefficients = [float(normalized_cc(s, e)[0]) for s, e in zip(shapes, events)]
    least_squares = 1.5 + np.median([np.log10(max(c, 1e-6) * np.linalg.norm(e) / np.linalg.norm(s))
                                     for c, s, e in zip(coefficients, shapes, events)])
    print(f"{label:>14s}: coefficients {np.round(coefficients, 2)}; your magnitude "
          f"{relative_magnitude(shapes, events, 1.5):.2f}, least-squares magnitude {least_squares:.2f} "
          "(template 1.5, events 10 times larger)")

**Explain (3-5 sentences):** What magnitudes did your function and the least-squares ratio give for the two kinds
of event, and why do they differ for the second kind? Which of the two measures would make the small events of a
swarm look systematically smaller than they are, and when? Why the median over channels rather than the mean? What
does a relative magnitude inherit from the template's magnitude?

## Part 5 · Two templates on three hours (about 120 min, then task 5)

Now the real records, with two templates from your own picks: the ML 3.9 (your M20 picks) and the ML 1.5 at
02:10:52 UTC (picks you make now). Each template has one window per channel: the P window on the vertical channel
starts `PRE_S` before your P pick and lasts `P_LEN_S`; the S windows on both horizontal channels start `PRE_S`
before your S pick and last `S_LEN_S`. With P and S at eight stations that is up to 24 channels per template.

**Pick the ML 1.5.** Use the picker E2 of M20 on the records around 02:10:52: P on the vertical channel of every
station where you can see it, and S on a horizontal channel where you can. The event is about 250 times smaller in
amplitude than the ML 3.9, so most onsets are small; switch on the causal band-pass of the picker. For templates
the polarity does not matter (leave it `?` when unsure); the uncertainty does: give each pick an honest one. Your
picks go to `SMALL_PICKS` (`work/picks/M22_ml15_picks.csv`) when you press Save.

In [ ]:
SMALL_ORIGIN = KMA_TIMES[SMALL_ID]
around_small = raw.slice(SMALL_ORIGIN - 30.0, SMALL_ORIGIN + 60.0).copy()
small_picker = st.widgets.phase_picker(around_small, origin=SMALL_ORIGIN, epicenter=(EV_LAT, EV_LON),
                                       csv_path=SMALL_PICKS, window=(-3.0, 20.0), fmin=2.0, fmax=15.0)
small_picker

**The network cross-correlation, step by step.** For template `T` with origin time `t0` (its KMA origin time):

1. On channel `c` the template window starts at sample `i_c`, the sample nearest to (pick - `PRE_S`) counted from
   the channel's first sample; the template's origin is sample `o`, nearest to `t0`. The channel's lag is
   `d_c = i_c - o` samples.
2. `normalized_cc(template_c, record_c)` gives one value per piece start `j`. A copy of the template earthquake
   with origin sample `q` shows up on channel `c` at `j = q + d_c`.
3. The network cross-correlation at sample `q` is the mean over all channels of the channel's value at
   `j = q + d_c`; a channel whose `q + d_c` falls outside its values adds 0 there. It has one value per sample of
   the records, and sample `q` is the time `T0 + q * DELTA`.
4. Your `mad_detections` with `K_MAD` and `MIN_GAP_S` gives the detections. At its own origin a template meets
   itself on every channel: a **self-detection** (glossary `self_detection`) with a value of 1, which proves
   nothing about the detector but checks your bookkeeping.

**The null test.** How many of your detections could be chance? A **null test** (glossary `null_test`) runs the
same method after deliberately breaking an expected relation. Reverse every record in time: its amplitudes and
frequency content remain, while the expected waveform and arrival alignment change. This is a useful diagnostic,
but some waveforms and coherent noise can still correlate; reversal does not certify an event-free control.
Correlate the same templates with the reversed records, build the network cross-correlation the same way and
count its peaks at or above the same threshold as for the real records (with `MIN_GAP_S`): those are false
candidates on this control, and their count per hour is the control's candidate rate. Do not equate it with a
measured false-alarm rate on ordinary records. Part 7 separates a null diagnostic from held-out validation.

### M22-T5 · Template matching on three hours · core · about 120 min

**Goal:** scan the three hours with your two templates, give every detection a relative magnitude, compare the
list with the KMA catalog, run the null test, and write two files.

**Given:** the prepared records (`prepared`, Part 0), your picks (`M20_PICKS` and `SMALL_PICKS`), the KMA times
(`KMA_TIMES`, `KMA_ML`) and your functions `normalized_cc`, `mad_detections`, `relative_magnitude` and
`picks_from_probability`.

**Contract:**
- Templates: for each of the two KMA events, from your picks of it at the stations of these records: P windows
  on the channel ending in Z, S windows on both channels ending in N and E, cut from `prepared` as in step 1;
  at least 3 stations per template.
- The network cross-correlation, the detections and the null test as described above (steps 1-4), with your
  functions. On the reversed records use the same template windows and lags, and count with your
  `picks_from_probability` at the real records' threshold.
- Relative magnitude of a detection at sample `q`: your `relative_magnitude` with the template windows and, on
  each channel, the `len(template)` samples of `prepared` that start at `q + d_c` (channels where they do not fit
  inside the record are left out), and the template's KMA magnitude.
- `results/M22_detections.csv`: one row per detection of either template, sorted by template, then time, with the
  columns `template` (the KMA id of the template's event), `time_utc` (ISO 8601: `T0 + q * DELTA`),
  `network_cc`, `magnitude` and `kma_event_id` (the id of the KMA event whose origin time lies within `MATCH_S`
  of `time_utc`, empty when there is none).
- `results/M22_template_matching.json` with the keys `dataset`, `band_hz` (`[FMIN_HZ, FMAX_HZ]`), `corners`,
  `taper_s`, `pre_s`, `p_len_s`, `s_len_s`, `k_mad`, `min_gap_s`, `match_s`, `detections_csv` (the path of the
  CSV file) and `templates`: a list with one entry per template holding `event_id`, `ml`, `origin_utc`,
  `picks_csv` (the path of the picks file it came from), `channels` (a list of `{"id": "<NET>.<STA>..<CHA>",
  "phase": "P" or "S", "start_utc": the time of sample i_c, "lag_samples": d_c}`), `median`, `mad`,
  `threshold`, `n_detections`, `null_threshold` (the threshold used on the reversed records) and
  `null_n_detections`.
- Set `tm_result = {"json": <path of the JSON file>, "ncc": normalized_cc, "detect": mad_detections,
  "magnitude": relative_magnitude, "network": {<KMA id>: (<network cross-correlation of the records>,
  <that of the reversed records>), ...}}`, with the two numpy arrays of each template (the cells after the
  check draw them).
- Example of a channel entry, with invented values: `{"id": "XX.STA1..HHZ", "phase": "P", "start_utc":
  "2031-04-05T06:07:08.090000Z", "lag_samples": 412}`.

**Done when** `st.check("M22-T5", answer=tm_result, dataset=DATASET)` passes. The check first runs your three
functions on a few made-up cases of tasks 2-4 again. Then it reads your two files, your picks files and your
records, and recomputes your network cross-correlations and the null test with ObsPy's `correlate_template`
(which your functions may not use): your windows must sit at your picks, and your two arrays, threshold,
detections, magnitudes, null count and KMA matches must follow from your own templates. It never compares with
a hidden list of detections and never prints its values.

**Stuck?** `st.hint("M22-T5", 1)`

In [ ]:
# Before you run: besides the two KMA events, how many detections do you expect from each template above 8 MAD in
# three hours, and of what magnitudes? Which template should find more small events, and why? Will the ML 3.9
# template give the ML 1.5 a magnitude of 1.5? How many detections do you expect on the reversed records?
st.predict("M22-T5", """

""")

In [ ]:
# Pick the ML 1.5 in the picker above and press Save (or type the picks, one call per pick:
#     small_picker.set_pick("XX.STA1", "P", <seconds after 02:10:52>, <uncertainty in s>, "?", channel="HHZ")
# and then small_picker.save()). Then build the two templates, scan the records, run the null test, write the two
# files and set tm_result. The steps, for each template (write small functions and call them for both):
# Step 1 (windows): read its picks file; for each pick at a station of `prepared`, take the channels it needs
#   (P: Z; S: N and E) and cut P_LEN_S or S_LEN_S seconds that begin at the sample i nearest to pick - PRE_S.
# Step 2 (lags): o = the sample nearest to the KMA origin time; each channel keeps d = i - o (step 1 of Part 5).
# Step 3 (the shifted mean): one array of zeros as long as the records; for each channel, normalized_cc of its
#   window with its whole record, and add the value at q + d to element q (only where q + d exists); divide the
#   sum by the number of channels.
# Step 4 (detections above the threshold): mad_detections on that array with K_MAD and MIN_GAP_S; a detection at
#   t seconds is the sample q = round(t / DELTA), the time T0 + q * DELTA; keep median, MAD and threshold.
# Step 5 (the reversed records): repeat step 3 with each record turned around ([::-1]), the same windows and lags;
#   count the peaks of picks_from_probability on it at the threshold of step 4 (not a new one).
# Step 6 (magnitudes): for each detection q, relative_magnitude of the windows against the pieces of `prepared`
#   that start at q + d (leave out channels whose piece runs past the end) with the template's KMA ML; the KMA
#   id is that of an origin time within MATCH_S, or empty.
# Step 7 (the files written): the CSV rows of both templates, then the JSON with every key of the Contract;
#   tm_result last, with both arrays (records, reversed) of each template under "network".
# YOUR CODE HERE
raise NotImplementedError("M22-T5: write your code here")

In [ ]:
st.check("M22-T5", answer=tm_result, dataset=DATASET)

Your detections, template by template, and what the null test found:

In [ ]:
summary = json.loads(Path(tm_result["json"]).read_text(encoding="utf-8"))
found_table = pd.read_csv(summary["detections_csv"], dtype={"template": str, "kma_event_id": str},
                          keep_default_na=False)
hours = (T1 - T0) / 3600.0
for entry in summary["templates"]:
    print(f"template {entry['event_id']} (ML {entry['ml']}), {len(entry['channels'])} channels at "
          f"{len({c['id'].split('.')[1] for c in entry['channels']})} stations: MAD {entry['mad']:.4f}, threshold "
          f"{entry['threshold']:.3f}; {entry['n_detections']} detections on the records, "
          f"{entry['null_n_detections']} on the reversed records ({entry['null_n_detections'] / hours:.2f} per hour)")
shown = found_table.assign(mads_above=[
    round((row.network_cc - entry["median"]) / entry["mad"], 1)
    for row in found_table.itertuples()
    for entry in summary["templates"] if entry["event_id"] == row.template])
print(shown.round({"network_cc": 3, "magnitude": 2}).to_string(index=False))

The same test at other thresholds: for each `k`, the detections in the records and in the reversed records, both
counted with your functions at the threshold that `mad_detections` gives on the real records.

In [ ]:
network = tm_result["network"]
null_rows = []
for entry in summary["templates"]:
    real_cc, null_cc = network[entry["event_id"]]
    for k_try in (5.0, 6.0, 7.0, 8.0, 9.0, 10.0):
        on_real, _, level_k = mad_detections(real_cc, DELTA, k_try, MIN_GAP_S)
        on_null, _ = picks_from_probability(null_cc, DELTA, level_k, MIN_GAP_S)
        null_rows.append({"template": entry["event_id"], "k": k_try, "threshold": round(level_k, 3),
                          "records": len(on_real), "reversed": len(on_null)})
print(pd.DataFrame(null_rows).pivot_table(index="k", columns="template", values=["records", "reversed"]))

The figure shows the network cross-correlation of each template over the three hours, with its threshold and
detections, and below it that of the reversed records with the same threshold.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(8.0, 7.5), sharex=True)
minutes = np.arange(len(network[MAINSHOCK_ID][0])) * DELTA / 60.0
for row_axes, entry in zip((axes[:2], axes[2:]), summary["templates"]):
    real_cc, null_cc = network[entry["event_id"]]
    for ax, values, label in ((row_axes[0], real_cc, "records"), (row_axes[1], null_cc, "reversed")):
        ax.plot(minutes, values, color="k", lw=0.3)
        ax.axhline(entry["threshold"], color="tab:red", lw=0.8, ls="--")
        ax.set_ylabel(f"ML {entry['ml']}\n{label}", fontsize=8)
    row_axes[1].set_ylim(row_axes[0].get_ylim())                  # one scale for the records and the reversed
    mine = found_table[found_table["template"] == entry["event_id"]]
    row_axes[0].plot([(UTCDateTime(t) - T0) / 60.0 for t in mine["time_utc"]], mine["network_cc"], "o",
                     mfc="none", color="tab:red", ms=6)
axes[-1].set_xlabel(f"Minutes after {T0.strftime('%H:%M:%S')} UTC")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "network_cc_three_hours.png", dpi=150)

Review the unmatched candidates yourself. A detection absent from KMA is not automatically a new earthquake.
The figure shows, for every detection that is not a KMA event, the S windows of
the ML 1.5 template at its three nearest stations next to the same windows of the detection, each scaled to its
own largest value.

In [ ]:
new_ones = found_table[(found_table["kma_event_id"] == "") & (found_table["template"] == SMALL_ID)]
small_entry = next(e for e in summary["templates"] if e["event_id"] == SMALL_ID)
s_channels = [c for c in small_entry["channels"] if c["phase"] == "S"][:6]
if len(new_ones):
    fig, axes = plt.subplots(len(s_channels), len(new_ones) + 1, figsize=(2.6 * (len(new_ones) + 1), 6.0),
                             squeeze=False, sharex=True)
    for col, start in enumerate([SMALL_ORIGIN] + [UTCDateTime(t) for t in new_ones["time_utc"]]):
        for row, ch in enumerate(s_channels):
            tr = prepared.select(id=ch["id"])[0]
            first = int(round((start - tr.stats.starttime) / DELTA)) + ch["lag_samples"]
            piece = tr.data[first - 100:first + int(S_LEN_S / DELTA) + 100]
            axes[row, col].plot(np.arange(piece.size) * DELTA - 1.0, piece / np.abs(piece).max(), color="k",
                                lw=0.5)
            axes[row, col].set_yticks([])
            if col == 0:
                axes[row, col].set_ylabel(ch["id"].split(".", 1)[1].replace("..", " "), fontsize=7)
        axes[0, col].set_title("template ML 1.5" if col == 0 else start.strftime("%H:%M:%S"), fontsize=8)
    for ax in axes[-1]:
        ax.set_xlabel("Seconds after the window start")
    fig.tight_layout()
    fig.savefig(Path(FIGURE_DIR) / "new_detections.png", dpi=150)
print(f"{len(new_ones)} detections of the ML 1.5 template are not in the KMA catalog")

**Explain (3-5 sentences):** How many detections did each template make, and which are not in the KMA catalog
(times, network values in MADs, magnitudes)? What magnitude did the ML 3.9 template give the ML 1.5, and the ML 1.5
template the ML 3.9; by how many units of log amplitude do the two events differ in our band, against the
catalog's 2.4? Which way would a lower corner frequency of the ML 3.9 move that difference in a 2-10 Hz band, does
your number fit, and what else could make it differ? Which template would you trust for the small events? How
many detections did the reversed records give at `K_MAD` and at lower `k`, and what does that say about your
threshold? Do the new events look like earthquakes in the waveform figure?

**Explorer E11 (optional, not checked): the threshold and the null test.** The explorer (a **widget**, glossary
`widget`, as in M20) draws the network cross-correlation of one template and that of the reversed records, with a
slider for `k` and one for the minimum gap. The threshold and the detections come from your `mad_detections`; the false detections on the reversed
records from your `picks_from_probability` at the same threshold. It computes neither itself: without your
functions it shows the curves only. Things to try: move `k` from 4 to 12 and watch the two counts; at which `k`
does the reversed record stop giving detections, and how many real detections are left there? Change the minimum
gap from 0.5 s to 20 s: which detections merge?

In [ ]:
# Optional explorer E11 (not checked): your threshold and detections on the ML 1.5 template's network
# cross-correlation, and the false detections on the reversed records.
small_real, small_null = network[SMALL_ID]
st.widgets.detection_explorer(small_real, DELTA, detect=mad_detections, peaks=picks_from_probability,
                              null_cc=small_null, starttime=T0, k=K_MAD, min_gap_s=MIN_GAP_S,
                              marks={f"ML {KMA_ML[e]:.1f}": t for e, t in KMA_TIMES.items()})

## Part 6 · A deep-learning detector and an associator (about 90 min, then task 6)

A deep-learning picker such as **PhaseNet** (glossary `phasenet`; Zhu & Beroza 2019) gives pick-probability
curves (glossary `pick_probability`) for P and S at every station (M21). On a continuous record every peak above
the **pick threshold** (glossary `pick_threshold`) is a **phase pick** (glossary `phase_pick`), and nothing says
which picks belong to the same earthquake. **Phase association** (glossary `phase_association`) groups them:
picks of one earthquake must fit one origin time and one set of **travel times** (glossary `travel_time`).
Production associators such as GaMMA (Zhu et al. 2022) and PyOcto (Münchmeyer 2024) search over places and times.
Here the place is known, the Sangju cluster, so a much simpler associator does: subtract from every pick the
predicted travel time from the KMA hypocenter to its station. The result is the pick's **back-projected origin
time** (glossary `back_projected_origin_time`): the origin time that pick points to. Picks of one earthquake near
the hypocenter point to nearly the same time; noise picks point anywhere. Picks whose back-projected times lie
within an **association window** (glossary `association_window`) of each other, at enough stations, make one
event. The window must allow for pick errors and for an earthquake a few kilometres from the KMA hypocenter.

### M22-T6 · An associator · core · about 60 min

**Goal:** write `associate(picks, travel_times, window_s, min_stations)`: events from picks whose back-projected
origin times agree.

**Given:** picks of many stations, and the predicted P and S travel times to each station from your `direct_time`
of M30.

**Contract:**
- Input: `picks`, a pandas DataFrame with the columns `station`, `phase` (`P` or `S`) and `time_s` (seconds after
  one common start time); `travel_times`, a dict `{(station, phase): seconds}`; `window_s` (s, > 0);
  `min_stations`, a whole number (1 or more).
- The back-projected origin time of a pick is `time_s - travel_times[(station, phase)]`. Picks whose
  `(station, phase)` has no travel time are left out.
- Go through the picks in order of their back-projected times. At each pick that is not yet part of an event,
  take it and every later pick not yet part of an event whose back-projected time is at most its own +
  `window_s`. If they come from at least `min_stations` different stations, they make an event (its origin time
  is the median of their back-projected times) and are now part of it; otherwise nothing changes and you go on
  with the next pick.
- Output: a pandas DataFrame with the columns `origin_s` (float), `n_stations` and `n_picks` (whole numbers), one
  row per event, sorted by `origin_s`; an empty DataFrame with these columns when there is no event. `picks` must
  not change.
- Example: picks `XX.STA1 P 12.0`, `XX.STA2 P 13.0`, `XX.STA3 S 16.4`, `XX.STA1 S 30.0` with the travel times
  2.0 (STA1 P), 3.2 (STA2 P), 6.1 (STA3 S), 3.5 (STA1 S), `window_s = 0.6` and `min_stations = 3` give one event:
  origin_s 10.0, 3 stations, 3 picks (back-projected 9.8, 10.0 and 10.3; the S of STA1 points to 26.5).
- Write it yourself: PyOcto, GaMMA and scikit-learn are not allowed here.

**Done when** `st.check("M22-T6", func=associate)` passes: on made-up picks of known events (picks that scatter
around the true arrivals, missing picks, stray picks, events at too few stations, picks exactly at the edge of
the window).

**Stuck?** `st.hint("M22-T6", 1)`

In [ ]:
# Before you code: in the example, what happens with window_s = 0.4, and what with min_stations = 2? PhaseNet's
# picks of the real records: will the associator find the two KMA events at 3 or more stations? Will it find the
# small events that your templates found? What will a lower pick threshold (0.1 instead of 0.3) add?
st.predict("M22-T6", """

""")

In [ ]:
# Your direct_time of module M30 (its task 1): paste it here, unchanged. Part 6 uses it after the check.
# YOUR CODE HERE
raise NotImplementedError("M22-T6: write your code here")

In [ ]:
def associate(picks, travel_times, window_s, min_stations):
    """Events of picks whose back-projected origin times lie within window_s, at min_stations or more stations.

    picks        : DataFrame with station, phase (P or S), time_s (s after a common start); not changed
    travel_times : dict {(station, phase): predicted travel time in s}
    window_s     : back-projected times of one event lie within this window (s)
    min_stations : smallest number of different stations of an event
    returns      : DataFrame with origin_s, n_stations, n_picks, sorted by origin_s
    """
    # Step 1: (back-projected time, station) for every pick with a travel time; sort by the time.
    # Step 2: walk through them; from each pick not yet in an event, gather the later free picks within window_s.
    # Step 3: enough stations: an event (median origin), and its picks are taken; else go on with the next pick.
    # YOUR CODE HERE
    raise NotImplementedError("M22-T6: write your code here")

In [ ]:
st.check("M22-T6", func=associate)

**The travel times.** Your `direct_time` gives the P and S travel times from the KMA hypocenter to each station
in `VELOCITY_MODEL`. For the KMA depth of 14 km all stations lie inside the crossover distances of Part 0, so
the direct waves are the first arrivals there. For an aftershock at 14.8 km only the farthest station is beyond
the S crossover, and close to a crossover the head wave leads the direct wave by a small part of a second (the two
arrive together at the crossover itself; your `head_wave` of M30 gives the number), far less than the association
window, so the associator, which only needs each pick to point within that window, still groups its picks. The
cell also shows how well these times fit your own picks of the ML 3.9: its back-projected origin times, station
by station.

In [ ]:
model = st.data.velocity_model(VELOCITY_MODEL)
travel = {}                                      # {(station, phase): seconds} from the KMA hypocenter
for row in stations.itertuples():
    for phase in ("P", "S"):
        travel[(row.station, phase)] = float(direct_time(row.epi_km, EV_DEPTH_KM, model, phase))
main_picks = m20_picks[m20_picks["station"].isin(stations["station"])]
fit = pd.DataFrame({"station": main_picks["station"], "phase": main_picks["phase"],
                    "travel_s": [travel[(s, p)] for s, p in zip(main_picks["station"], main_picks["phase"])],
                    "back_projected_s": [UTCDateTime(t) - KMA_TIMES[MAINSHOCK_ID] - travel[(s, p)]
                                         for s, p, t in zip(main_picks["station"], main_picks["phase"],
                                                            main_picks["time_utc"])]})
print(f"travel times in {VELOCITY_MODEL} from {EV_DEPTH_KM:g} km depth; your ML 3.9 picks, back-projected "
      "(s after the KMA origin):")
print(fit.round(3).to_string(index=False))
print(f"spread of the back-projected times: {fit['back_projected_s'].max() - fit['back_projected_s'].min():.2f} s")

Your associator on made-up picks at these eight stations, with these travel times: five made-up events, picks that
scatter by up to 0.3 s around the predicted arrivals, about a third of the picks missing, and 40 stray picks at
random times (`synth_detect.pick_table`). How many events does it find, for several `min_stations`?

In [ ]:
made_up_picks, made_up_truth = synth_detect.pick_table(np.random.default_rng(31), list(stations["station"]), travel,
                                                        [600.0, 1500.0, 1520.0, 2400.0, 3300.0], jitter_s=0.3,
                                                        p_miss=0.35, n_stray=40, duration_s=3600.0)
print(f"{len(made_up_picks)} made-up picks; the five events got picks at "
      f"{[len(s) for s in made_up_truth['stations']]} stations")
for needed in (2, 3, 4, 5):
    found = associate(made_up_picks, travel, ASSOC_WINDOW_S, needed)
    true_ones = sum(any(abs(o - t) < 1.0 for t in made_up_truth["origin_s"]) for o in found["origin_s"])
    print(f"min_stations {needed}: {len(found)} events, {true_ones} of them within 1 s of a made-up origin time")

**Explain (3-5 sentences):** How many of the five made-up events did your associator find for each `min_stations`,
and did stray picks make events of their own? Two of the made-up events are only 20 s apart: did they stay apart,
and what would make them merge? Why must the association window be wider than the scatter of the picks and the
misfit of the travel times (look at the spread of your own ML 3.9 picks above), and what goes wrong if it is much
wider?

**PhaseNet on three hours, three runs.** The production detector is SeisBench's PhaseNet (Woollam et al. 2022)
with the `stead` weights of M21 (Mousavi et al. 2019), checked against the course's checksums. Its `classify`
finds the picks itself (the peaks of the probability curves above a threshold, like your
`picks_from_probability`). Three runs: as installed, with `PN_THRESHOLD` for P and S; again, unchanged, to see
whether the output is **deterministic** (glossary `deterministic_computation`); and with the one changed setting
`PN_THRESHOLD_CHANGED`. The raw records go in (PhaseNet scales each window itself). Each run takes a few seconds
on one core.

In [ ]:
import torch

torch.set_num_threads(1)                         # inference on one core (shared-server rule)
try:
    import seisbench
    import seisbench.models as sbm
except ImportError as exc:
    raise ImportError(f"SeisBench cannot load its models in this environment ({exc}). Run 'sgtl doctor' in a "
                      "terminal and tell your mentor.") from None
pinned = {}
for line in st.data.repo_path("install", "checksums.txt").read_text(encoding="utf-8").splitlines():
    parts = line.split()
    if len(parts) == 3 and parts[0].startswith("phasenet/"):
        pinned[parts[0]] = parts[1]
for kind in ("json", "pt"):
    rel = f"phasenet/{PN_WEIGHTS[0]}.{kind}.v{PN_WEIGHTS[1]}"
    path = Path(seisbench.cache_model_root) / rel
    if not path.is_file():
        raise FileNotFoundError(f"{path} is missing. In a terminal: bin/sgtl-install seisbench --apply")
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{rel}: SHA-256 {digest[:16]}... "
          f"{'matches the course pin' if pinned.get(rel) == digest else 'DOES NOT MATCH the pin: tell your mentor'}")
phasenet = sbm.PhaseNet.from_pretrained(PN_WEIGHTS[0], version_str=PN_WEIGHTS[1])
phasenet.eval()


def phasenet_picks(threshold):
    """PhaseNet's picks on the raw three hours (P and S at one threshold), as a table in seconds after T0."""
    output = phasenet.classify(raw, P_threshold=threshold, S_threshold=threshold)
    table = pd.DataFrame([{"station": ".".join(p.trace_id.split(".")[:2]), "phase": p.phase,
                           "time_s": float(p.peak_time - T0), "probability": float(p.peak_value)}
                          for p in output.picks], columns=["station", "phase", "time_s", "probability"])
    return table.sort_values(["time_s", "station", "phase"], ignore_index=True)


pn_first = phasenet_picks(PN_THRESHOLD)
pn_rerun = phasenet_picks(PN_THRESHOLD)
pn_changed = phasenet_picks(PN_THRESHOLD_CHANGED)
print(f"threshold {PN_THRESHOLD}: {len(pn_first)} picks; rerun identical: {pn_rerun.equals(pn_first)}")
print(f"threshold {PN_THRESHOLD_CHANGED}: {len(pn_changed)} picks")
counts = pd.concat({f"threshold {PN_THRESHOLD}": pn_first.groupby(["phase", "station"]).size(),
                    f"threshold {PN_THRESHOLD_CHANGED}": pn_changed.groupby(["phase", "station"]).size()}, axis=1)
print(counts.fillna(0).astype(int).T.to_string())
away = np.array([all(abs(T0 + t - k) > 30.0 for k in KMA_TIMES.values()) for t in pn_changed["time_s"]], dtype=bool)
far = pn_changed[(pn_changed["phase"] == "P").to_numpy() & away]
print(f"\nP picks at threshold {PN_THRESHOLD_CHANGED} more than 30 s from both KMA origin times:")
print(far.assign(time_utc=[str(T0 + t) for t in far["time_s"]])[["time_utc", "station", "probability"]]
      .round(2).to_string(index=False))

Your associator on the picks of both thresholds, and all lists side by side: the KMA catalog, your network
coincidence of Part 1 (with `MIN_STATIONS`), your template detections of Part 5 (either template) and PhaseNet with
your associator. Two entries of different lists are the same event when their times differ by less than 3 s; a
network-coincidence detection counts for an event when it covers its time as in Part 1.

In [ ]:
pn_events = {}
for label, table in (("PhaseNet " + str(PN_THRESHOLD), pn_first), ("PhaseNet " + str(PN_THRESHOLD_CHANGED),
                                                                  pn_changed)):
    found = associate(table, travel, ASSOC_WINDOW_S, ASSOC_MIN_STATIONS)
    pn_events[label] = [T0 + float(t) for t in found["origin_s"]]
    print(f"{label}: {len(found)} events")
    print(found.assign(origin_utc=[str(T0 + float(t)) for t in found["origin_s"]]).round(2).to_string(index=False))

lists = {"KMA": list(KMA_TIMES.values()),
         "templates": [UTCDateTime(t) for t in sorted(set(found_table["time_utc"]))],
         **pn_events}
merged = []
for name, times in lists.items():
    for when in times:
        for event in merged:
            if abs(event["time"] - when) < 3.0:
                event[name] = "x"
                break
        else:
            merged.append({"time": when, name: "x"})
for event in merged:
    event["coincidence"] = "x" if any(covers(start, end, event["time"] - T0) for start, end, _ in network_found) else ""
comparison = pd.DataFrame(merged).fillna("").sort_values("time", ignore_index=True)
comparison["time"] = [t.strftime("%H:%M:%S.%f")[:-4] for t in comparison["time"]]
print(comparison.to_string(index=False))

**Explain (3-5 sentences, not checked):** Was the rerun identical, and why does that matter before you change the
threshold? How many picks and how many associated events did each threshold give, and which events are they? Which
events did only the templates find, which only PhaseNet, and why can each method miss what the other finds (think
of where the events are and how similar they are to the templates)? How many of the network-coincidence
detections of Part 1 does any other method confirm? Which list would you hand to a colleague who wants every
earthquake of these three hours, and what would you tell them about its completeness?

## Part 7 · One candidate per event, and an honest validation protocol (about 120 min)

A **template bank** (glossary `template_bank`) is the collection of templates searched together, with a version,
source events, picks, channels and processing settings. One earthquake can match several bank members. Counting
every row of `found_table` as another earthquake would count some earthquakes twice. **Cross-template
de-duplication** (glossary `cross_template_deduplication`) combines those reports while retaining their evidence.
This is different from M05's statistical declustering: you are removing duplicate reports, not aftershocks.

A large correlation is not an origin-time uncertainty or a probability that a detection is real. Here the
ranking score is `(network_cc - median) / mad`, using each template's own median and unscaled MAD from T5.
It is a bookkeeping score, not a common false-alarm probability. The representative is the strongest match;
every supporting template ID and every original row remain available. Different templates may give different
relative magnitudes, so the candidate table does not silently choose a magnitude or a hypocenter.

Choose a merge tolerance from timing resolution, then test close pairs: too wide a tolerance can merge two
real earthquakes. The exercise uses a fixed window starting at a group's earliest match. A chain of matches
cannot keep extending that window. Production tools also separate per-template trigger suppression from
cross-template merging; see [EQcorrscan's Party.decluster documentation](https://eqcorrscan.readthedocs.io/en/latest/submodules/core.match_filter.party.html).
This exercise defines its own fixed-anchor grouping, rather than reproducing that tool's method.

**Held-out evaluation** (glossary `held_out_evaluation`) means evaluating a frozen detector on records kept
separate from template choice and tuning. Split data by time before inspecting results, not by randomly
splitting overlapping windows of the same waveform. Template supports, filter/transient margins, calibration
and evaluation must not overlap. Overlapping processing chunks still need enough overlap to cover template
moveout and window length; remove duplicate edge detections afterward and count their exposure only once.
The [EQcorrscan matched-filter documentation](https://eqcorrscan.readthedocs.io/en/latest/submodules/core.match_filter.template.html)
explains why channel offsets can otherwise lose detections near chunk edges. Its MAD convention also differs
from this lesson's median plus unscaled MAD: verify formulas and sum/mean normalization before transferring a
numerical threshold between implementations.

You have already inspected all three teaching hours, and T5 used their full correlation curves to set its
thresholds. The protocol below is therefore explicitly a **retrospective teaching plan**, not evidence of blind
evaluation. In each research project, reserve fresh evaluation times before tuning. Record any **prior
exposure** (glossary `prior_exposure`) to those data; an inspected evaluation period must be replaced or reported
honestly as retrospective.

Your protocol must address all of these, in your own words:

1. Freeze the bank, channel rules, filter, score, threshold rule, merge tolerance and review rules after
   calibration. Adding a template creates a new bank version and more opportunities for chance matches;
   remeasure the rate for the whole bank, rather than quoting one template's rate.
2. Estimate a **false-alarm rate** (glossary `false_alarm_rate`) from independently reviewed controls and their
   usable exposure: confirmed false candidates divided by live hours after gaps, missing-channel rules and
   edge guards. Unmatched candidates are not known false alarms; inspect waveforms and compare independent
   evidence. Reversal and independent channel shifts are complementary diagnostics, not certified event-free
   controls. Zero false candidates in a short control does not prove a zero rate; under a Poisson model with
   independent, stationary counts, zero in H hours gives the one-sided 95 percent upper bound `-log(0.05) / H`.
   State when those assumptions fail, and report counts and exposure as well as the rate.
3. Measure **injection and recovery** (glossary `injection_recovery`) with a frozen detector on separate
   background windows: known insertion times and amplitude scales, many trials across noise periods and station
   subsets, a declared time tolerance and one recovered candidate per insertion. Run the identical uninjected
   backgrounds too, so a pre-existing candidate is not credited as a recovery. Keep failed insertions in the
   denominator; report counts and uncertainty per amplitude/noise bin, with ambiguous close pairs flagged.
   Exact copies of bank templates are an optimistic test: include held-out waveform families or controlled
   waveform changes, and do not translate amplitude into magnitude without a checked calibration.
4. Explain **selection bias** (glossary `selection_bias`): a bank built from cataloged, high-SNR earthquakes
   favors similar sources, paths and station epochs. Review an independent sample and use synthetic signals
   that differ from the bank. Self-detection and a few small detections establish neither completeness nor a
   new magnitude of completeness. Treat new candidates as candidates until reviewed, picked and located.
5. Transfer the method separately to Suncheon and to the Sangju project: inventory available
   continuous records and response/channel epochs; choose and document each bank from that project's data;
   reserve its own unseen calibration/evaluation periods; size the first run for the shared server. The 2019
   teaching templates, thresholds, counts and relative magnitudes are not a result for either project.


### M22-T7 · Merge a template bank and write its validation protocol · core · about 120 min

**Goal:** write `merge_template_matches(matches, window_s)`, apply it to T5's matches and save the candidate table
and the validation protocol. Retain T5's unmerged files. Read the protocol with your mentor before a project run.

**Given:** `found_table`, `summary`, `T0`, `T1`, the parameters at the top, and `json`/`hashlib` for records.
The merger itself needs ordinary Python; do not call a clustering or template-matching library.

**Contract (function):**
- `matches`: a list of `(template_id, time_s, score)` tuples, with non-empty string IDs and finite numbers;
  `time_s` is seconds after the record start. `window_s` is finite and nonnegative. Do not change the list.
- Visit matches in increasing time order. A group contains the earliest remaining match and every subsequent
  match whose time is at most that earliest time plus `window_s`, including the boundary. Start a new group
  with the next remaining match. The group's anchor never moves as members join.
- Its representative has the highest score; ties choose the earliest time, then the alphabetically first ID.
- Return a list in time order of `(time_s, score, representative_id, template_ids, n_matches)`, where
  `template_ids` is a sorted tuple of distinct supporting IDs and `n_matches` counts all raw rows in the group,
  including repeated rows of one template. Empty input returns `[]`.
- Example: `[('A', 0.0, 8.0), ('B', 0.75, 9.0), ('C', 1.5, 10.0)]` at 1 s gives
  `[(0.75, 9.0, 'B', ('A', 'B'), 2), (1.5, 10.0, 'C', ('C',), 1)]`; the first group does not grow to 1.5 s.

**Contract (artifacts):**
- Build input rows from T5's unrounded CSV using `(template, UTCDateTime(time_utc) - T0,
  (network_cc - template_median) / template_mad)`. Each template's MAD must be positive.
- Save `results/M22_candidates.csv` with columns `time_s,time_utc,score,representative,templates,n_matches`;
  `templates` is a JSON list of the supporting IDs. Every row must come from your merger at `DEDUP_WINDOW_S`.
- Save `results/M22_validation_protocol.json` with exactly these keys:
  `dataset`, `exercise_status` (`"retrospective_teaching_plan"`), `bank_version` (non-empty text),
  `template_ids` (sorted IDs), `score_definition` (`"mads_above_median"`), `merge_window_s`,
  `calibration_window_s`, `evaluation_window_s` (each `[start, end]` after T0), `guard_s`,
  `calibration_live_hours`, `evaluation_live_hours`, `matching_json_sha256`, `matches_csv_sha256`,
  and the seven written entries `threshold_freeze`, `control_and_exposure`, `injection_recovery`,
  `selection_bias`, `prior_exposure`, `suncheon_transfer`, `sangju2026_transfer`.
- Use `CALIBRATION_WINDOW_S`, `EVALUATION_WINDOW_S`, `VALIDATION_GUARD_S` for the rehearsal. Windows must stay
  inside the record after the guard at both edges, be separated by at least twice the guard, and avoid each
  template's origin by the guard. The guard must be at least the largest absolute channel lag plus the longest
  template window plus the taper. For these gap-free teaching records live hours equal window seconds / 3600;
  real project exposure must also subtract invalid times. Fingerprints are SHA-256 of file bytes.
- Write at least 12 words per written entry, no `[write here]` markers. These are plans, not invented measured
  performance: describe controls, denominators, comparisons, uncertainty, frozen choices and honest limits.
- Set `bank_validation = {"matching_json": tm_result["json"], "candidates_csv": <candidate CSV path>,
  "validation_json": <protocol JSON path>}`.

**Done when** `st.check("M22-T7", func=merge_template_matches, report=bank_validation, dataset=DATASET)` passes:
made-up banks with known groups, ties and close pairs; then your candidate file against your own verified
merger, and the protocol's provenance, windows, exposure and required entries. The software checks structure
and consistency, not whether the written plan is scientifically adequate: explain it to your mentor.

**Stuck?** `st.hint("M22-T7", 1)`

In [ ]:
# Before you code: should two templates detecting one earthquake make two events? In the example above, why must
# a third match not extend the first group's window? What can an exact-template injection fail to reveal?
st.predict("M22-T7", """

""")

In [ ]:
def merge_template_matches(matches, window_s):
    """Combine fixed-anchor groups; retain representative, distinct template IDs and raw match counts.

    matches : list of (template_id, time_s, score), not changed
    window_s : nonnegative grouping tolerance in seconds
    returns : list of (time_s, score, representative_id, sorted template_ids tuple, n_matches)
    """
    # YOUR CODE HERE
    raise NotImplementedError("M22-T7: write your code here")


# Write the candidate CSV and your protocol, then set bank_validation, as the Contract describes.
# YOUR CODE HERE
raise NotImplementedError("M22-T7: write your code here")

In [ ]:
st.check("M22-T7", func=merge_template_matches, report=bank_validation, dataset=DATASET)

**Explain (3-5 sentences):** Compare the raw match count with the merged candidate count and inspect every group
supported by more than one template. Which close pair could your tolerance wrongly merge, and what evidence
would separate it? Explain why the protocol is retrospective, what fresh data each project must reserve, how
you would measure false alarms and recovery, and why neither self-detection nor zero null detections proves
completeness. Be ready to show the raw provenance and defend your plan in the oral check.

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M22")

### Going further (optional, not checked)

1. Make a third template from one of your new detections (its windows at the same lags) and scan the three hours
   again. Does it find events that the two KMA templates missed?
2. Shuffle the channels instead of reversing them: give every channel's network contribution a different random
   shift of at least a minute, so that no earthquake can line up across channels. Does this null test agree with
   the time reversal?
3. Repeat Part 5 with a band of 3-12 Hz and with 5-15 Hz. Which detections survive every band, and how do the
   relative magnitudes of the two KMA events change?
4. Run PyOcto (Münchmeyer 2024), which the course environment installs, on PhaseNet's picks with your travel
   times, and compare its events with those of your associator.

Next: M31 locates earthquakes from picks, and M41 measures differential times by cross-correlation, the same
measure of likeness you used here.